# Meta Muse Glimmer 30B を Google Colab で動かす

Metaの **Muse Glimmer 30B** を、公式GGUF + `llama.cpp` を使ってGoogle Colab上でローカル実行します。

このNotebookでは、公開直後のモデルであることを考慮し、`transformers + bitsandbytes` ではなく、対応の早い **llama.cpp** を利用します。

## 方針

- Hugging Face上の公式GGUFリポジトリからモデルを取得
- Colab上で最新版 `llama.cpp` をCUDA対応でビルド
- 量子化済みGGUFをGPUへオフロード
- OpenAI互換APIとして `llama-server` を起動
- 日本語で動作確認
- GradioによるシンプルなチャットUI

## 推奨環境

**Google Colab Pro / 24GB級GPU以上を推奨します。**

Muse Glimmer 30BのQ4級GGUFは約18GB前後あるため、T4 16GBなど無料版ColabのGPUでは、全層GPUロードは基本的に厳しいです。

初回は安定性を優先し、context lengthを8192 tokensに設定しています。


In [10]:
# =========================================
# コード1 実行環境の確認と必要パッケージ
# =========================================
!nvidia-smi -L || echo "No GPU"
!python -V

%pip -q install -U huggingface_hub gradio requests

import sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPUランタイムを有効にしてください。")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

props = torch.cuda.get_device_properties(0)
print("GPU memory: %.1f GB" % (props.total_memory / 1024**3))


GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-d36bc064-540d-e0eb-8e0f-92d331920363)
Python 3.12.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 79.5 MB/s eta 0:00:00
Python: 3.12.13
PyTorch: 2.11.0+cu128
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory: 95.0 GB


In [11]:
# =========================================
# コード2 Google Driveとキャッシュ設定
# =========================================
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/LocalLLM")
MODEL_DIR = PROJECT_DIR / "Muse_Glimmer_30B"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("MODEL_DIR:", MODEL_DIR)

usage = shutil.disk_usage(MODEL_DIR)
print("free space: %.1f GB" % (usage.free / 1024**3))

if usage.free < 30 * 1024**3:
    print("WARNING: 30GB以上の空き容量を推奨します。")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MODEL_DIR: /content/drive/MyDrive/Colab Notebooks/LocalLLM/Muse_Glimmer_30B
free space: 160.5 GB


In [12]:
# =========================================
# コード3 llama.cpp をCUDA対応でビルド
# =========================================
from pathlib import Path
import subprocess

LLAMA_DIR = Path("/content/llama.cpp")

if not LLAMA_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/ggml-org/llama.cpp.git",
         str(LLAMA_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(LLAMA_DIR), "pull"], check=True)

subprocess.run(
    [
        "cmake",
        "-S", str(LLAMA_DIR),
        "-B", str(LLAMA_DIR / "build"),
        "-DGGML_CUDA=ON",
        "-DLLAMA_CURL=ON",
        "-DCMAKE_BUILD_TYPE=Release",
    ],
    check=True,
)

subprocess.run(
    [
        "cmake", "--build", str(LLAMA_DIR / "build"),
        "--config", "Release",
        "-j", "2",
    ],
    check=True,
)

LLAMA_SERVER = LLAMA_DIR / "build" / "bin" / "llama-server"

if not LLAMA_SERVER.exists():
    raise FileNotFoundError(f"llama-server not found: {LLAMA_SERVER}")

print("llama-server:", LLAMA_SERVER)


llama-server: /content/llama.cpp/build/bin/llama-server


In [13]:
# =========================================
# コード4 公式GGUFを自動検出してダウンロード
# =========================================
from huggingface_hub import list_repo_files, hf_hub_download
from pathlib import Path

REPO_ID = "meta-models/Muse-Glimmer-30B-GGUF"

files = list_repo_files(REPO_ID)

print("GGUF files:")
for f in files:
    if f.lower().endswith(".gguf"):
        print(" ", f)

model_candidates = [
    f for f in files
    if f.lower().endswith(".gguf")
    and "mmproj" not in f.lower()
    and "dflash" not in f.lower()
]

priority_keywords = [
    "kquant-dynamic",
    "q4_k_xl",
    "q4_k_m",
    "q4",
]

MODEL_FILE = None
for key in priority_keywords:
    matches = [f for f in model_candidates if key in f.lower()]
    if matches:
        MODEL_FILE = sorted(matches)[0]
        break

if MODEL_FILE is None and model_candidates:
    MODEL_FILE = sorted(model_candidates)[0]

if MODEL_FILE is None:
    raise RuntimeError("利用可能なモデルGGUFを検出できませんでした。")

print()
print("Selected model:", MODEL_FILE)

MODEL_PATH = Path(
    hf_hub_download(
        repo_id=REPO_ID,
        filename=MODEL_FILE,
        local_dir=str(MODEL_DIR),
    )
)

print("MODEL_PATH:", MODEL_PATH)
print("size: %.2f GB" % (MODEL_PATH.stat().st_size / 1024**3))


GGUF files:
  Muse-Glimmer-30B-KQuant-17GB-Q4_K_M.gguf
  Muse-Glimmer-30B-KQuant-Dynamic-Q4_K_XL.gguf
  dflash-Muse-Glimmer-30B-Q4_K_M.gguf
  dflash-kquant.gguf
  mmproj-Muse-Glimmer-30B-Q4_K_M.gguf
  mmproj-kquant.gguf
  muse-glimmer-30B-kquant-17gb.gguf
  muse-glimmer-30B-kquant-dynamic.gguf

Selected model: Muse-Glimmer-30B-KQuant-Dynamic-Q4_K_XL.gguf
MODEL_PATH: /content/drive/MyDrive/Colab Notebooks/LocalLLM/Muse_Glimmer_30B/Muse-Glimmer-30B-KQuant-Dynamic-Q4_K_XL.gguf
size: 18.30 GB


## GPUメモリに合わせた設定

24GB級GPUでは、Q4級のMuse Glimmer 30Bをほぼ全層GPUへ載せることを狙います。

無料版Colabなどで16GB GPUしか割り当てられなかった場合は、`GPU_LAYERS` を小さくするとCPUへ一部を逃がせますが、速度はかなり低下します。

初回は `CONTEXT_SIZE=8192` としています。モデル自体はより長いcontextを扱えますが、長いcontextほどKV cacheが増えるため、Colabではまず短い設定から試す方が安全です。


In [14]:
# =========================================
# コード5 llama-serverを起動
# 空いているポートを自動取得する版
# =========================================
import subprocess
import time
import requests
import socket

CONTEXT_SIZE = 8192
GPU_LAYERS = 999

# -----------------------------------------
# 空いているTCPポートを自動取得
# -----------------------------------------
def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

PORT = find_free_port()

print("Using port:", PORT)

# -----------------------------------------
# 念のため既存の llama-server を停止
# -----------------------------------------
subprocess.run(
    ["pkill", "-9", "-f", "llama-server"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(2)

SERVER_LOG = "/content/muse_glimmer_server.log"

cmd = [
    str(LLAMA_SERVER),
    "-m", str(MODEL_PATH),
    "--host", "127.0.0.1",
    "--port", str(PORT),
    "-c", str(CONTEXT_SIZE),
    "-ngl", str(GPU_LAYERS),
    "--flash-attn", "on",
    "--jinja",
    "--parallel", "1",
]

print("Starting llama-server...")
print(" ".join(cmd))

log_fp = open(SERVER_LOG, "w")

server_process = subprocess.Popen(
    cmd,
    stdout=log_fp,
    stderr=subprocess.STDOUT,
)

base_url = f"http://127.0.0.1:{PORT}"

# -----------------------------------------
# server ready を待つ
# -----------------------------------------
for i in range(180):

    if server_process.poll() is not None:
        log_fp.flush()

        print("\n--- llama-server log ---")
        print(open(SERVER_LOG).read()[-8000:])

        raise RuntimeError(
            "llama-server terminated unexpectedly."
        )

    try:
        r = requests.get(
            base_url + "/health",
            timeout=2,
        )

        if r.status_code == 200:
            print("llama-server is ready.")
            break

    except requests.exceptions.RequestException:
        pass

    if i % 10 == 0:
        print("waiting...", i)

    time.sleep(2)

else:
    log_fp.flush()

    print("\n--- llama-server log ---")
    print(open(SERVER_LOG).read()[-8000:])

    raise TimeoutError(
        "llama-server did not become ready."
    )

print()
print("base_url:", base_url)

!nvidia-smi


Using port: 58785
Starting llama-server...
/content/llama.cpp/build/bin/llama-server -m /content/drive/MyDrive/Colab Notebooks/LocalLLM/Muse_Glimmer_30B/Muse-Glimmer-30B-KQuant-Dynamic-Q4_K_XL.gguf --host 127.0.0.1 --port 58785 -c 8192 -ngl 999 --flash-attn on --jinja --parallel 1
waiting... 0
llama-server is ready.

base_url: http://127.0.0.1:58785
Wed Aug 12 23:24:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+=====================

In [15]:
# =========================================
# コード6 日本語で動作確認
# =========================================
import requests

def muse_generate(
    user_text,
    system_text="あなたは日本語で簡潔に答える親切なアシスタントです。",
    max_tokens=512,
    temperature=0.7,
):
    payload = {
        "model": "muse-glimmer",
        "messages": [
            {"role": "system", "content": system_text},
            {"role": "user", "content": user_text},
        ],
        "temperature": float(temperature),
        "max_tokens": int(max_tokens),
        "stream": False,
    }

    r = requests.post(
        base_url + "/v1/chat/completions",
        json=payload,
        timeout=600,
    )
    r.raise_for_status()

    return r.json()["choices"][0]["message"]["content"].strip()

reply = muse_generate(
    "日本語で1文だけ自己紹介してください。",
    max_tokens=256,
)

print(reply)


In [16]:
# =========================================
# コード7 GradioによるシンプルなローカルLLMチャットUI
# Gradioのバージョン差異に対応
# =========================================
import gradio as gr
import requests
import inspect

print("Gradio:", gr.__version__)


# -----------------------------------------
# Gradioのバージョン差異を吸収
# -----------------------------------------
def make_messages_chatbot(**kwargs):
    params = inspect.signature(gr.Chatbot).parameters

    if "type" in params:
        kwargs["type"] = "messages"

    return gr.Chatbot(**kwargs)


def gr_chat(history, user_msg):
    history = history or []
    user_msg = (user_msg or "").strip()

    if not user_msg:
        return history, "", history

    messages = [
        {
            "role": "system",
            "content": "あなたは日本語で簡潔に答える親切なアシスタントです。",
        }
    ]

    # 直近の会話だけ利用
    for msg in history[-8:]:
        if isinstance(msg, dict):
            if msg.get("role") in ("user", "assistant"):
                messages.append(msg)

    messages.append(
        {
            "role": "user",
            "content": user_msg,
        }
    )

    try:
        payload = {
            "model": "muse-glimmer",
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 512,
            "stream": False,
        }

        r = requests.post(
            base_url + "/v1/chat/completions",
            json=payload,
            timeout=600,
        )

        r.raise_for_status()

        reply = (
            r.json()["choices"][0]["message"]["content"]
            .strip()
        )

    except Exception as e:
        reply = f"{type(e).__name__}: {e}"

    new_history = history + [
        {
            "role": "user",
            "content": user_msg,
        },
        {
            "role": "assistant",
            "content": reply,
        },
    ]

    return new_history, "", new_history


with gr.Blocks(title="Muse Glimmer 30B Local Chat") as demo:

    gr.Markdown(
        "## Muse Glimmer 30B — Local GGUF Chat"
    )

    chatbot = make_messages_chatbot(
        show_label=False,
    )

    chat_state = gr.State([])

    user_box = gr.Textbox(
        placeholder="質問を入力してください。",
        label="",
        lines=3,
    )

    with gr.Row():
        send_btn = gr.Button(
            "Send",
            variant="primary",
        )

        clear_btn = gr.Button(
            "Clear"
        )

    send_btn.click(
        gr_chat,
        inputs=[
            chat_state,
            user_box,
        ],
        outputs=[
            chat_state,
            user_box,
            chatbot,
        ],
        queue=False,
    )

    user_box.submit(
        gr_chat,
        inputs=[
            chat_state,
            user_box,
        ],
        outputs=[
            chat_state,
            user_box,
            chatbot,
        ],
        queue=False,
    )

    clear_btn.click(
        lambda: ([], "", []),
        outputs=[
            chat_state,
            user_box,
            chatbot,
        ],
        queue=False,
    )


print(
    "WARNING: share=True で"
    "一時的な公開URLを作成します。"
)

demo.launch(
    share=True,
    inline=True,
    debug=False,
)


Gradio: 6.23.1
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e0a1790f1179f955d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## トラブルシューティング

### `CUDA out of memory`

Muse Glimmer 30BはQ4級でも約18GB前後あるため、16GB GPUでは全層GPUロードできない可能性が高いです。

コード5の

```python
GPU_LAYERS = 999
```

を例えば、

```python
GPU_LAYERS = 30
```

などへ下げると、一部layerをCPUへ置くことができます。ただし生成速度は大きく低下します。

### L4 24GBでもOOMする場合

まずcontextを下げます。

```python
CONTEXT_SIZE = 4096
```

また、モデルファイルとしてより小さいQ3系GGUFを選ぶ方法もあります。

### `llama-server` が起動しない

Muse Glimmerは公開直後のモデルなので、古いllama.cppでは未対応の可能性があります。このNotebookでは毎回最新版をclone/pullしてbuildする構成にしています。

### 初回ダウンロード

モデルGGUFが約18GB級あるため、初回のダウンロードには時間がかかります。Google Driveへ保存するため、2回目以降は再利用できます。


### `couldn't bind HTTP server socket`

このv2では8080番ポートを固定せず、OSに空きポートを自動選択させます。
以前の `llama-server` や別のWebサーバと競合しにくい構成です。

### `Chatbot.__init__() got an unexpected keyword argument 'type'`

Gradioのバージョンによって `gr.Chatbot` の引数が異なる場合があります。
このv2では `inspect.signature()` で `type` 引数の有無を確認してから設定するため、
`type="messages"` を受け付けない環境でも動作するようにしています。
